# Matthias: what the snow data tell us

**The question is simple: are there fewer days with enough natural snow now than before?**

We have two sources: readings taken at real measuring stations, and estimates from a computer weather model. We check the estimates against the readings before using them for all the ski resorts.

**Start with the short table and before/after chart below.** The detailed checks come afterwards. The saved data are already included: Run All works without new weather downloads by default.

For the short explanation, open [Matthias’s snow work: the simple version](../docs/matthias_simple_explanation.md).

## 1. Load the project

In Codespaces, select the Python kernel and Run All. Install `requirements.txt` in the terminal first if packages are missing. The helper code is in `src/snow_validation.py` and `src/snow_reporting.py`.

In [2]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

start = Path.cwd().resolve()
root = next((p for p in [start, *start.parents]
             if (p / "data/processed/meteoswiss_station_winters.csv").is_file()), None)
if root is None:
    raise FileNotFoundError("Open the notebook inside the sci-alpi project folder.")
sys.path.insert(0, str(root / "src"))
from snow_validation import load_observations, download_models, pair_sources, export_results
from snow_reporting import build_report
from snow_plain_summary import build_plain_summary
from snow_method_check import compare_selections

observations = load_observations(root)
print(f"Measured input: {observations.station_id.nunique()} stations, {len(observations)} complete station winters.")

## Start here: the actual measurements

We count days with at least 30 cm of snow, then compare the average for ten older winters with ten recent winters. These results use **only real Swiss station measurements**, independently of the computer snow estimates.

“Fewer days” is the older average minus the recent average. Positive means fewer recent snow days.

In [3]:
measured_summary = build_plain_summary(root)
readable = measured_summary[["elevation_group", "n_stations", "days_past", "days_recent", "lost_days"]].copy()
readable["elevation_group"] = readable["elevation_group"].map({
    "all_stations": "All selected Swiss stations",
    "below_1500m": "Below 1,500 m",
    "at_least_1500m": "At least 1,500 m",
})
readable.columns = ["Station group", "Stations", "Older average (days)", "Recent average (days)", "Fewer days"]
display(readable.round(1))
print("These are 20 selected Swiss stations above 1,000 m, not all Alpine resorts.")

,Station group,Stations,Older average (days),Recent average (days),Fewer days
0,All selected Swiss stations,20,78.3,63.0,15.4
1,"Below 1,500 m",11,55.3,33.1,22.3
2,"At least 1,500 m",9,106.5,99.5,7.0


![Actual measured snow days before and after](../docs/figures/snow_measured_before_after.png)

The lower stations in this sample lost more snow days on average. The result describes this station sample. It does not establish the difference for all resorts or prove that elevation caused the change.

---

## Details: can we trust the computer estimates?

The rest of the notebook checks the estimates and the resort inputs. You can present the measured result above and the finding that the estimates need care; the original four-country snow analysis is still open.

## 2. Match model and measurements

We request ERA5-Land at the **station coordinates and elevation**, for the same dates as the measurements. The actual returned grid point can be some distance away; it is retained in the table and shown on the map.

MeteoSwiss `hto000d0` measures snow depth at **06:00 UTC**. Therefore the primary model count also uses 06:00 UTC, with the same 30 cm threshold and Dec 1–Apr 15 season. The model's daily-maximum count is included as a sensitivity check; it is the indicator used in the original resort pilot.

Leave `RUN_VALIDATION_DOWNLOAD = False` to use the committed 400 model seasons. Changing it to True downloads missing raw seasons using the local cache. It can take several minutes. A rate-limit response stops the batch, keeps completed caches and preserves existing published data.

In [4]:
RUN_VALIDATION_DOWNLOAD = False

if RUN_VALIDATION_DOWNLOAD:
    models = download_models(root)
else:
    models = pd.read_csv(root / "data/processed/station_model_winters.csv")

pairs = pair_sources(observations, models)
metrics, changes, trend = export_results(root, pairs)
print(f"Matched one-to-one: {len(pairs)} station winters. No station-winter keys lost.")
display(pairs[["station_id", "winter", "period", "observed_days_06utc", "model_days_06utc", "error_days", "grid_distance_km"]].head(10))

,station_id,winter,period,observed_days_06utc,model_days_06utc,error_days,grid_distance_km
0,ABO,1986,past,63,121,58,12.412203
1,ABO,1987,past,112,120,8,12.412203
2,ABO,1988,past,57,137,80,12.412203
3,ABO,1989,past,17,136,119,12.412203
4,ABO,1990,past,2,79,77,12.412203
5,ABO,1991,past,10,136,126,12.412203
6,ABO,1992,past,36,137,101,12.412203
7,ABO,1993,past,8,136,128,12.412203
8,ABO,1994,past,6,129,123,12.412203
9,ABO,1995,past,29,109,80,12.412203


## 3. How close are the counts?

- **Bias** = model minus measured snow days. Positive means the model counts too many days.
- **MAE** = average absolute count error, in days per winter.
- **RMSE** = root mean squared count error, in days per winter; large errors receive more weight.
- **r** = descriptive correlation. A positive correlation can coexist with large bias.
- **Daily-max extra days** = how much the model count increases when using the daily maximum instead of 06:00.

The 400 rows repeat 20 stations. They are not 400 independent stations; no pooled significance test is reported.

In [5]:
display(metrics.round(3))

,period,n_station_winters,n_stations,bias_days,mae_days,rmse_days,pearson_r,mean_daily_max_extra_days,model_full_season_fraction,observed_full_season_fraction
0,all,400,20,54.212,56.438,73.132,0.280,0.288,0.632,0.082
1,past,200,20,46.815,49.795,65.260,0.295,0.240,0.640,0.070
2,recent,200,20,61.610,63.080,80.235,0.272,0.335,0.625,0.095


## 4. Does the model reproduce the change between decades?

For each station, average its ten past winters and its ten recent winters separately. **Lost days = past mean minus recent mean.** Positive means fewer recent snow days, negative means more.

The correlation of changes uses **one period difference per station (n=20)**. Its p-value is nominal: it assumes independent station changes, whereas nearby stations share weather. Treat it as exploratory, alongside the error and direction diagnostics.

In [6]:
display(changes[["station_name", "elev_m", "observed_days_past", "observed_days_recent", "observed_lost_days", "model_lost_days", "loss_error_days", "same_direction"]].round(2))
display(trend.round(4))

,station_name,elev_m,observed_days_past,observed_days_recent,observed_lost_days,model_lost_days,loss_error_days,same_direction
0,Adelboden,1321.0,34.0,23.5,10.5,-6.6,-17.1,False
1,Arosa,1878.0,113.9,116.2,-2.3,2.3,4.6,False
2,Bivio,1856.0,112.5,96.9,15.6,3.7,-11.9,True
3,Bosco/Gurin,1486.0,112.3,65.2,47.1,-0.9,-48.0,False
4,Château-d'Oex,1028.0,30.9,13.0,17.9,-0.8,-18.7,False
5,Chaumont,1136.0,39.0,11.6,27.4,8.7,-18.7,True
6,Davos,1594.0,100.1,104.4,-4.3,3.5,7.8,False
7,Disentis,1197.0,57.1,28.6,28.5,-0.7,-29.2,False
8,Gadmen,1190.0,88.4,75.7,12.7,0.9,-11.8,True
9,Grimsel Hospiz,1980.0,129.3,130.4,-1.1,-0.1,1.0,True


,n_stations,mean_observed_lost_days,mean_model_lost_days,mean_loss_error_days,loss_rmse_days,loss_pearson_r,loss_pearson_p_nominal,same_direction_stations
0,20,15.38,0.585,-14.795,20.0229,-0.1863,0.4316,7


## Follow-up: does choosing a closer model point help?

We tried the closest possible model point instead of the original land selection. Everything else stayed the same. The check uses all 20 stations, but only the two pilot winters (1995 and 2025), giving 40 matched cases.

This is a small method check, not a new ten-year-period climate result. The closest-point option did not improve this test. The provider also does not mark snow depth for elevation correction, so entering a base or summit altitude does not establish snow at that exact height. Sources and details are in [the method check](../docs/snow_method_check.md).

In [7]:
nearest_pilot = pd.read_csv(root / "data/processed/station_model_nearest_pilot.csv")
selection_cases, selection_metrics = compare_selections(root, nearest_pilot)
display(selection_metrics[["selection", "n_station_winters", "bias_days", "mae_days", "rmse_days", "mean_grid_distance_km"]].round(2))
print("Use the same 40 cases to compare settings. These are not 20-winter trend estimates.")

,selection,n_station_winters,bias_days,mae_days,rmse_days,mean_grid_distance_km
0,land,40,49.42,52.32,64.66,6.64
1,nearest,40,53.25,56.15,66.92,3.73


## 5. Review the expanded resort sample and recreate figures

The new resort file contains 300 rows, but source URLs can repeat within and across countries. Coordinates can be present and still be wrong. This audit preserves all input rows and flags duplicate URLs, cross-border entries, obviously out-of-scope coordinates and a difference between the geocoding cache and final features.

The rectangle 43–49° N, 4–17° E is only an automatic screening rule. Passing it does not establish Alpine membership or a verified base-area coordinate. Failing it can mean a valid non-Alpine resort or a geocoding error. The source data are kept intact.

Figures and a plain-English findings report are regenerated from the saved tables.

In [8]:
audit = build_report(root, pairs)
print(f"Expanded input: {len(audit)} rows, {audit.url.nunique()} distinct source URLs.")
print(f"Rows needing location or identity review: {int(audit.requires_location_or_identity_review.sum())}.")
display(audit.loc[audit.requires_location_or_identity_review, ["source_csv_row", "resort_name", "country", "latitude", "longitude", "review_reasons"]].head(15))
print("Full audit: data/processed/snow_resort_input_review.csv")
print("Findings and next steps: docs/matthias_next_steps.md")
print("Figures: docs/figures/")

,source_csv_row,resort_name,country,latitude,longitude,review_reasons
0,2,Les Portes du Soleil – Morzine/​Avoriaz/​Les G...,CH,46.179200,6.773100,duplicate_url; cross_border_url; cache_coordin...
1,3,Zermatt/​Breuil-Cervinia/​Valtournenche – Matt...,CH,45.960986,7.716347,duplicate_url; cross_border_url
4,6,Arosa Lenzerheide,CH,46.754888,9.566024,duplicate_url
5,7,Ischgl/​Samnaun – Silvretta Arena,CH,46.969374,10.339581,duplicate_url; cross_border_url
17,19,4 Vallées – Verbier/​La Tzoumaz/​Nendaz/​Veyso...,CH,47.513685,2.997351,outside_alpine_screening_box
75,77,Via Lattea – Sestriere/​Sauze d’Oulx/​San Sica...,IT,37.493112,15.076460,duplicate_url; cross_border_url; outside_alpin...
76,78,Zermatt/​Breuil-Cervinia/​Valtournenche – Matt...,IT,45.954152,7.699504,duplicate_url; cross_border_url
97,99,Espace San Bernardo – La Rosière/​La Thuile,IT,43.499510,-1.491083,duplicate_url; cross_border_url; outside_alpin...
121,123,Alto Sangro – Roccaraso/​Rivisondoli,IT,41.848291,14.080164,outside_alpine_screening_box
132,134,San Domenico – Alpe Ciamporino,IT,42.023851,13.750775,outside_alpine_screening_box


## Figures

![Matched snow-day counts](../docs/figures/snow_validation_counts.png)

![Observed and modelled changes](../docs/figures/snow_validation_changes.png)

![Station and grid locations](../docs/figures/snow_validation_stations_map.png)

The following figure describes the original ten reference locations only. It does not establish a country-wide climate effect or validate the resort risk classification.

![Original pilot](../docs/figures/snow_pilot_period_change.png)

## What remains for Matthias's climate analysis

1. Review model agreement and its ability to reproduce the observed period change. A universal subtraction of average bias is not a validated correction.
2. Resolve the expanded roster's duplicate ski areas, cross-border country rule, Alpine sample scope and base-area coordinates with Lorenzo.
3. Download 20 complete winters for each approved unique location using the agreed indicator and a resumable cache. Publish the full table only after completeness checks pass.
4. Produce the full-sample change chart and map, then investigate country and elevation differences (SQ1). The pilot's small, unverified sample is descriptive.
5. Hand the approved snow features and limitations to Luca for the price and closure analyses.

This Swiss manual-station subset is above 1,000 m and is not a homogenised climate series. Check site histories and representativeness before attributing long-term changes to climate or extrapolating the validation to other countries and lower-altitude resorts.

Sources: [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api), [MeteoSwiss manual station data](https://opendatadocs.meteoswiss.ch/a-data-groundbased/a5-manual-precipitation-stations), [MeteoSwiss parameter metadata](https://data.geo.admin.ch/ch.meteoschweiz.ogd-nime/ogd-nime_meta_parameters.csv). Map boundaries: [Natural Earth, public domain](https://www.naturalearthdata.com/about/terms-of-use/). Weather data: ECMWF/Copernicus ERA5-Land via Open-Meteo (CC BY 4.0).

## Follow-up: the resort roster

The original source rows are preserved. This separate roster combines duplicate ski areas, records cross-border systems separately and excludes reviewed resorts outside the Alps. Source access points identify representative locations; they are not certified measurements at the minimum ski elevation.


In [9]:
from resort_review import build_roster

roster = build_roster(root)
print(f"{len(roster)} distinct areas; {int(roster.include_in_alpine_candidate_set.sum())} Alpine candidates.")
display(roster.groupby(["country_for_analysis", "scope_decision"]).size().rename("Areas").to_frame())
large_moves = roster.loc[roster.scope_decision.eq("in_alps") & roster.coordinate_change_km.gt(30)]
display(large_moves[["resort_name", "original_latitude", "original_longitude", "latitude", "longitude", "coordinate_change_km"]].round(3))
print("260 identities have not been individually rechecked. Base-elevation matching and the snow method remain unvalidated.")


Areas
country_for_analysis scope_decision                  
AT                   in_alps                        1
                     not_individually_reviewed     73
CH                   in_alps                        2
                     not_individually_reviewed     70
FR                   in_alps                        6
                     not_individually_reviewed     49
                     outside_alps                  15
IT                   in_alps                        1
                     not_individually_reviewed     68
                     outside_alps                   3
cross_border         in_alps                        5

,resort_name,original_latitude,original_longitude,latitude,longitude,coordinate_change_km
1,4 Vallées – Verbier/​La Tzoumaz/​Nendaz/​Veyso...,47.514,2.997,46.079,7.215,358.441
73,Espace Lumière – Pra Loup/​Val d’Allos,48.896,2.151,44.370,6.600,606.955
74,Espace San Bernardo – La Rosière/​La Thuile,43.500,-1.491,45.626,6.864,702.505
145,Le Grand Domaine – Valmorel/​Doucy/​Celliers/​...,48.619,-2.267,45.461,6.441,746.910
153,Les 3 Vallées – Val Thorens/​Les Menuires/​Mér...,-12.755,45.229,45.432,6.619,7541.036
157,Les Brasses,44.615,2.918,46.169,6.464,326.275
159,Les Houches/​Saint-Gervais – Prarion/​Bellevue...,49.356,-1.262,45.895,6.782,714.647
213,Puy-Saint-Vincent,45.772,3.084,44.818,6.486,286.432
228,San Domenico – Alpe Ciamporino,42.024,13.751,46.249,8.188,646.080
281,Via Lattea – Sestriere/​Sauze d’Oulx/​San Sica...,37.493,15.076,45.026,6.861,1081.955


## Follow-up: can another weather model help?

This screening compares the same 20 stations in winters 1995 and 2021. It does not replace the original study periods or validate a long-term trend. Missing model values are unavailable data, never zero snow.


In [10]:
from snow_candidates import compare_candidates

candidate_models = pd.read_csv(root / "data/processed/station_model_candidates.csv")
candidate_cases, candidate_metrics = compare_candidates(root, candidate_models)
display(candidate_metrics[["model", "n_station_winters", "screening_winters", "bias_days", "mae_days", "rmse_days"]].round(2))
availability = pd.read_csv(root / "data/processed/snow_model_availability.csv")
display(availability[["model", "station_id", "winter", "returned_hours", "non_null_hours", "availability_status"]])
print("CERRA improves the small screening but still has substantial error; the public API lacks the recent data we need.")


,model,n_station_winters,screening_winters,bias_days,mae_days,rmse_days
0,era5_land,40,1995;2021,31.20,34.65,46.25
1,cerra,40,1995;2021,14.05,27.80,39.13


,model,station_id,winter,returned_hours,non_null_hours,availability_status
0,cerra,ABO,1995,3264,3264,complete
1,cerra,ABO,2021,3264,3264,complete
2,cerra,ABO,2025,3264,0,missing_not_zero
3,era5,ABO,1995,3264,0,missing_not_zero


## What is ready next?

The direct Copernicus request plan is in `data/config/cerra_cds_download_plan.json`. It needs a private CDS account/key and acceptance of the dataset licence. No authenticated retrieval has been tested here. Start by inspecting one recent file, then validate it before bulk downloads or resort risk labels.

See [the expansion status](../docs/snow_expansion_status.md) for exact commands and remaining limitations. This notebook's default run makes no downloads.
